# Speech Emotion Router (SER): Non-Linear Tabular Modeling & Performance Ceilings

In the previous phase ([`02_baseline_modeling.ipynb`](./02_baseline_modeling.ipynb)), our flat multinomial Logistic Regression benchmark established that static MFCC summaries provide strong demographic separation but struggle significantly with intra-gender emotional ambiguity.

This phase investigates whether non-linear decision boundaries can resolve overlapping affective clusters before we transition to continuous 2D spectral representations.

**Phase Objectives & Experimental Scope:**

* **Non-Linear Tabular Benchmark:** Evaluate a Support Vector Classifier with a Radial Basis Function (RBF) kernel on the 26-dimensional static MFCC summary matrix to test the limits of non-linear geometric boundaries.

* **Hyperparameter Calibration:** Conduct a structured grid search across regularization ($C$) and kernel bandwidth ($\gamma$) to balance boundary flexibility against speaker overfitting.

* **Leak-Free Grouped Validation:** Enforce the established 5-fold `StratifiedGroupKFold` harness grouped by `actor_id` with in-pipeline scaling to ensure evaluation on unseen speakers.

* **Performance Ceiling Assessment:** Compare out-of-fold metrics and confusion patterns directly against the linear baseline to determine whether static temporal summaries inherently constrain valence recognition.

In [ ]:
# Standard library
from pathlib import Path
import pprint
from typing import Any, cast

# Data handling and numerical arrays
import joblib
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
from matplotlib.container import BarContainer
import seaborn as sns

# Scikit-learn: pipelines and preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Scikit-learn: validation and evaluation metrics
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    cross_val_predict,
    cross_validate,
)

# Scikit-learn: estimators
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# Plotting style configuration
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100